# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** c23_design_space_optimizer     
**Author:** Jasper Cluistra   
**Last Updated:** 2026-03-19
### Design space optimization script
**Goal:** Optimize the structure by iterating over structural performance, circular performance, and total cost.
**Inputs:**
*   Search space definition
*   Trained surrogate model
*   Material input dataset

**Outputs:**
*   Optimized design candidates
*   Ranked solution set

# OPTIMIZER

In [1]:
# =============================================================================
# c30_ga_algorithm_v1.py — (μ+λ) Evolution Strategy for Truss Geometry
# =============================================================================
#
# Optimises continuous geometry parameters (node positions / spans) using a
# (μ+λ) Evolution Strategy with adaptive step sizes (self-adaptive σ).
#
# Why (μ+λ) ES over standard GA:
#   - Expensive evaluations (5–30s each): μ=10, λ=20 means only 20 evaluations
#     per generation — parents are never re-evaluated (elitism is free).
#   - Continuous search space: Gaussian mutation is theoretically grounded for
#     real-valued parameters. Crossover helps escape local optima.
#   - Self-adaptive σ: each individual carries its own step size vector that
#     evolves alongside the design parameters — no manual σ tuning required.
#   - (μ+λ) selection (parents compete with offspring) gives stronger elitism
#     than (μ,λ) (offspring-only), which is appropriate when evaluations are
#     expensive and you can't afford to lose good solutions.
#
# Operator summary:
#   Selection  : tournament selection from (parents ∪ offspring) pool
#   Crossover  : intermediate recombination (arithmetic mean of two parents)
#   Mutation   : Gaussian perturbation with self-adaptive σ per parameter
#   σ update   : log-normal update rule (standard ES self-adaptation)
#
# Usage:
#   from c30_ga_algorithm_v1 import EvolutionStrategy, ESConfig
#
#   cfg = ESConfig(mu=10, lam=20, n_generations=50)
#   es  = EvolutionStrategy(
#       search_space   = optimizer_search_space,
#       evaluate_fn    = evaluate_design_candidate_wrapped,
#       config         = cfg,
#   )
#   result = es.run()
#
# Requires from notebook:
#   evaluate_design_candidate()  — the full pipeline evaluator
#   optimizer_search_space       — dict of {param_name: (lower, upper)}
#   SURROGATE_BUNDLE, df_input_stock, FIXED_NORMALIZATION_CONSTANTS, GA_CONFIG

from __future__ import annotations

import time
import random
import warnings
from copy import deepcopy
from dataclasses import dataclass, field
from typing import Any, Callable

import numpy as np

# =============================================================================
# CONFIGURATION DATACLASS
# =============================================================================

@dataclass
class ESConfig:
    """
    All hyperparameters for the (μ+λ) Evolution Strategy.

    Sensible defaults for expensive evaluations (5–30s per call):
        mu=10, lam=20 → 20 evaluations per generation
        n_generations=50 → ~1000 total evaluations → ~6 hours at 20s/eval
        For faster iteration during development: mu=5, lam=10, n_generations=20
    """
    # Population
    mu:            int   = 10      # number of parents kept each generation
    lam:           int   = 20      # number of offspring generated each generation

    # Termination
    n_generations: int   = 50      # maximum generations
    budget:        int   = None    # optional: hard cap on total evaluations

    # Crossover
    crossover_prob: float = 0.7    # probability of applying recombination
                                   # (vs cloning a single parent)

    # Self-adaptive mutation (log-normal σ update)
    sigma_init:    float = 0.1     # initial step size as fraction of param range
    sigma_min:     float = 1e-4    # floor — prevents σ collapsing to zero
    tau:           float = None    # global learning rate (None = 1/sqrt(n_params))
    tau_prime:     float = None    # local  learning rate (None = 1/sqrt(2*n_params))

    # Tournament selection
    tournament_size: int = 3       # k in k-tournament; higher = more selective

    # Restart
    stagnation_limit: int = 15     # restart if best fitness unchanged for this
                                   # many generations (None to disable)
    n_restarts_max:   int = 3      # max restarts before giving up

    # Penalty
    penalty_fitness: float = 1e6   # fitness assigned to infeasible / failed evals

    # Logging
    log_every:     int   = 1       # print summary every N generations
    verbose:       bool  = True


# =============================================================================
# INDIVIDUAL
# =============================================================================

class Individual:
    """
    One candidate design.

    Attributes
    ----------
    params : dict[str, float]
        Design parameters — one float per dimension, within search space bounds.
    sigma  : np.ndarray [n_params]
        Self-adaptive step sizes, one per parameter. Evolved alongside params.
    fitness : float
        Objective value (lower = better). None until evaluated.
    eval_result : dict
        Full output from evaluate_design_candidate() for post-processing.
    """

    __slots__ = ("params", "sigma", "fitness", "eval_result", "generation")

    def __init__(
        self,
        params:      dict[str, float],
        sigma:       np.ndarray,
        fitness:     float = None,
        eval_result: dict  = None,
        generation:  int   = 0,
    ):
        self.params      = params
        self.sigma       = sigma
        self.fitness     = fitness
        self.eval_result = eval_result
        self.generation  = generation

    def is_evaluated(self) -> bool:
        return self.fitness is not None

    def __repr__(self) -> str:
        f = f"{self.fitness:.4f}" if self.fitness is not None else "?"
        return f"Individual(fitness={f}, gen={self.generation})"


# =============================================================================
# EVOLUTION STRATEGY
# =============================================================================

class EvolutionStrategy:
    """
    (μ+λ) Evolution Strategy with self-adaptive Gaussian mutation.

    Parameters
    ----------
    search_space  : dict[str, tuple[float, float]]
        {param_name: (lower_bound, upper_bound)} for each design variable.
    evaluate_fn   : Callable[[dict], float]
        Function that takes design_params dict and returns scalar fitness.
        Lower = better. Should return config.penalty_fitness on failure.
        Wrap evaluate_design_candidate() with _make_evaluate_fn() below.
    config        : ESConfig
    seed          : int | None
    """

    def __init__(
        self,
        search_space: dict[str, tuple[float, float]],
        evaluate_fn:  Callable[[dict], float],
        config:       ESConfig = None,
        seed:         int      = None,
    ):
        self.search_space = search_space
        self.evaluate_fn  = evaluate_fn
        self.config       = config or ESConfig()
        self.param_names  = list(search_space.keys())
        self.n_params     = len(self.param_names)
        self.bounds_lo    = np.array([search_space[k][0] for k in self.param_names])
        self.bounds_hi    = np.array([search_space[k][1] for k in self.param_names])
        self.param_range  = self.bounds_hi - self.bounds_lo

        if seed is not None:
            np.random.seed(seed)
            random.seed(seed)

        # Learning rates for self-adaptive σ (ES theory defaults)
        cfg = self.config
        n   = self.n_params
        self.tau       = cfg.tau       or 1.0 / np.sqrt(n)
        self.tau_prime = cfg.tau_prime or 1.0 / np.sqrt(2 * n)

        # State
        self.population:    list[Individual] = []
        self.history:       list[dict]       = []   # one entry per generation
        self.best_ever:     Individual | None = None
        self.generation:    int               = 0
        self.n_evals:       int               = 0
        self.n_restarts:    int               = 0
        self.stagnation:    int               = 0

        print(
            f"[ES] Initialised: μ={cfg.mu} λ={cfg.lam} "
            f"n_generations={cfg.n_generations} n_params={n}\n"
            f"     τ={self.tau:.4f}  τ'={self.tau_prime:.4f}  "
            f"σ_init={cfg.sigma_init:.4f}  σ_min={cfg.sigma_min:.6f}\n"
            f"     Expected evaluations: "
            f"{cfg.mu + cfg.lam * cfg.n_generations} "
            f"(initial {cfg.mu} + {cfg.lam}×{cfg.n_generations} offspring)"
        )

    # -------------------------------------------------------------------------
    # PUBLIC API
    # -------------------------------------------------------------------------

    def run(self) -> dict[str, Any]:
        """
        Run the full evolution strategy.

        Returns
        -------
        result : dict
            best_individual  — Individual with lowest fitness found
            best_fitness     — float
            best_params      — dict[str, float]
            best_eval_result — full pipeline output dict
            history          — list of per-generation summaries
            n_evals          — total evaluations performed
            n_generations    — generations completed
            n_restarts       — number of restarts triggered
        """
        cfg        = self.config
        t_start    = time.time()
        budget     = cfg.budget or int(1e9)

        # Initialise population
        self.population = self._initialise_population(cfg.mu)
        self._evaluate_population(self.population)

        if cfg.verbose:
            print(f"\n[ES] Initial population evaluated ({cfg.mu} individuals)")
            self._print_generation_summary(0)

        for gen in range(1, cfg.n_generations + 1):
            if self.n_evals >= budget:
                print(f"[ES] Budget of {budget} evaluations reached at generation {gen}.")
                break

            self.generation = gen

            # Generate offspring
            offspring = self._generate_offspring(cfg.lam)

            # Evaluate offspring (parents are NOT re-evaluated — free elitism)
            self._evaluate_population(offspring)

            # (μ+λ) selection: best μ from parents ∪ offspring
            combined        = self.population + offspring
            self.population = self._select(combined, cfg.mu)

            # Track best ever
            gen_best = min(self.population, key=lambda ind: ind.fitness)
            if self.best_ever is None or gen_best.fitness < self.best_ever.fitness:
                self.best_ever  = deepcopy(gen_best)
                self.stagnation = 0
            else:
                self.stagnation += 1

            # Log
            self._record_history(gen)
            if cfg.verbose and gen % cfg.log_every == 0:
                self._print_generation_summary(gen)

            # Stagnation restart
            if (cfg.stagnation_limit is not None
                    and self.stagnation >= cfg.stagnation_limit
                    and self.n_restarts < cfg.n_restarts_max):
                self._restart()

        elapsed = time.time() - t_start
        print(
            f"\n[ES] Finished in {elapsed/60:.1f} min  |  "
            f"{self.n_evals} evaluations  |  "
            f"{self.generation} generations  |  "
            f"{self.n_restarts} restarts\n"
            f"[ES] Best fitness: {self.best_ever.fitness:.6f}"
        )

        return {
            "best_individual":  self.best_ever,
            "best_fitness":     self.best_ever.fitness,
            "best_params":      self.best_ever.params,
            "best_eval_result": self.best_ever.eval_result,
            "history":          self.history,
            "n_evals":          self.n_evals,
            "n_generations":    self.generation,
            "n_restarts":       self.n_restarts,
        }

    # -------------------------------------------------------------------------
    # INITIALISATION
    # -------------------------------------------------------------------------

    def _initialise_population(self, n: int) -> list[Individual]:
        """Sample n individuals uniformly at random within bounds."""
        population = []
        sigma_init = self.config.sigma_init * self.param_range

        for _ in range(n):
            params_arr = self.bounds_lo + np.random.uniform(0, 1, self.n_params) * self.param_range
            params     = dict(zip(self.param_names, params_arr))
            sigma      = sigma_init.copy()
            population.append(Individual(params=params, sigma=sigma, generation=0))

        return population

    # -------------------------------------------------------------------------
    # OPERATORS
    # -------------------------------------------------------------------------

    def _recombine(self, p1: Individual, p2: Individual) -> tuple[np.ndarray, np.ndarray]:
        """
        Intermediate recombination: offspring inherits arithmetic mean of
        parents for both params and sigma.

        Why intermediate (not discrete) recombination:
        - Continuous parameters → the mean of two valid parents is also valid
          within bounds (after clipping).
        - Averages σ vectors too, giving a conservative initial step size that
          is then refined by mutation.
        """
        arr1 = np.array([p1.params[k] for k in self.param_names])
        arr2 = np.array([p2.params[k] for k in self.param_names])
        child_arr   = 0.5 * (arr1 + arr2)
        child_sigma = 0.5 * (p1.sigma + p2.sigma)
        return child_arr, child_sigma

    def _mutate(self, params_arr: np.ndarray, sigma: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """
        Self-adaptive Gaussian mutation (ES standard).

        σ update (log-normal rule):
            σ_i' = σ_i * exp(τ' * N(0,1) + τ * N_i(0,1))
            where τ' is the global learning rate, τ is the local learning rate,
            N(0,1) is shared across all σ_i, N_i(0,1) is individual per σ_i.

        Parameter update:
            x_i' = x_i + σ_i' * N_i(0,1)

        The log-normal update ensures σ stays positive and can both increase
        and decrease — allowing the step size to self-tune during evolution.
        """
        # Update σ first (before applying to params — ES convention)
        global_noise = np.random.randn()
        local_noise  = np.random.randn(self.n_params)
        new_sigma    = sigma * np.exp(
            self.tau_prime * global_noise + self.tau * local_noise
        )
        new_sigma    = np.maximum(new_sigma, self.config.sigma_min)

        # Perturb parameters
        new_params_arr = params_arr + new_sigma * np.random.randn(self.n_params)

        # Clip to bounds (reflection would be theoretically cleaner but
        # clipping is simpler and works well in practice for box constraints)
        new_params_arr = np.clip(new_params_arr, self.bounds_lo, self.bounds_hi)

        return new_params_arr, new_sigma

    def _generate_offspring(self, n: int) -> list[Individual]:
        """Generate n offspring via recombination + mutation."""
        cfg      = self.config
        offspring = []

        for _ in range(n):
            # Select parent(s)
            if len(self.population) >= 2 and np.random.rand() < cfg.crossover_prob:
                p1, p2       = random.sample(self.population, 2)
                params_arr, sigma = self._recombine(p1, p2)
            else:
                p1           = random.choice(self.population)
                params_arr   = np.array([p1.params[k] for k in self.param_names])
                sigma        = p1.sigma.copy()

            # Mutate
            params_arr, sigma = self._mutate(params_arr, sigma)
            params            = dict(zip(self.param_names, params_arr))

            offspring.append(Individual(
                params     = params,
                sigma      = sigma,
                generation = self.generation,
            ))

        return offspring

    # -------------------------------------------------------------------------
    # SELECTION
    # -------------------------------------------------------------------------

    def _select(self, pool: list[Individual], n: int) -> list[Individual]:
        """
        Tournament selection: run n tournaments of size k, each returning
        the winner (lowest fitness). Used for (μ+λ) survivor selection.

        Tournament selection is preferred over truncation (top-n) here because:
        - It maintains more diversity — worse individuals occasionally survive.
        - It avoids premature convergence when several individuals have very
          similar fitness values near the optimum.
        """
        k        = min(self.config.tournament_size, len(pool))
        selected = []
        for _ in range(n):
            tournament = random.sample(pool, k)
            winner     = min(tournament, key=lambda ind: ind.fitness)
            selected.append(deepcopy(winner))
        return selected

    # -------------------------------------------------------------------------
    # EVALUATION
    # -------------------------------------------------------------------------

    def _evaluate_population(self, population: list[Individual]) -> None:
        """Evaluate each unevaluated individual in the population in-place."""
        to_eval = [ind for ind in population if not ind.is_evaluated()]
        n       = len(to_eval)

        if n == 0:
            return

        for i, ind in enumerate(to_eval):
            t0 = time.time()
            try:
                fitness, eval_result = self.evaluate_fn(ind.params)
            except Exception as exc:
                warnings.warn(f"[ES] Evaluation failed: {exc}")
                fitness     = self.config.penalty_fitness
                eval_result = {"status": "PENALIZED", "reason": str(exc)}

            ind.fitness     = fitness
            ind.eval_result = eval_result
            self.n_evals   += 1
            elapsed         = time.time() - t0

            if self.config.verbose:
                status = eval_result.get("status", "?") if eval_result else "?"
                print(
                    f"  [{self.n_evals:4d}] gen={self.generation} "
                    f"ind={i+1}/{n}  fitness={fitness:.4f}  "
                    f"status={status}  ({elapsed:.1f}s)"
                )

    # -------------------------------------------------------------------------
    # RESTART
    # -------------------------------------------------------------------------

    def _restart(self) -> None:
        """
        Partial restart: keep the best μ//2 individuals, reseed the rest.
        Resets σ for reseeded individuals to initial values.
        Preserves the best solution found so far.
        """
        cfg          = self.config
        n_keep       = cfg.mu // 2
        sorted_pop   = sorted(self.population, key=lambda ind: ind.fitness)
        keep         = sorted_pop[:n_keep]
        new_randoms  = self._initialise_population(cfg.mu - n_keep)

        self.population  = keep + new_randoms
        self.stagnation  = 0
        self.n_restarts += 1

        print(
            f"\n[ES] Restart {self.n_restarts}/{cfg.n_restarts_max} "
            f"at generation {self.generation}  "
            f"(kept {n_keep} elites, reseeded {cfg.mu - n_keep})"
        )

        # Evaluate new random individuals
        self._evaluate_population(new_randoms)

    # -------------------------------------------------------------------------
    # LOGGING
    # -------------------------------------------------------------------------

    def _record_history(self, gen: int) -> None:
        fitnesses = [ind.fitness for ind in self.population]
        sigmas    = np.array([ind.sigma for ind in self.population])
        entry     = {
            "generation":    gen,
            "n_evals":       self.n_evals,
            "best_fitness":  float(np.min(fitnesses)),
            "mean_fitness":  float(np.mean(fitnesses)),
            "worst_fitness": float(np.max(fitnesses)),
            "std_fitness":   float(np.std(fitnesses)),
            "mean_sigma":    float(sigmas.mean()),
            "stagnation":    self.stagnation,
            "best_ever":     self.best_ever.fitness if self.best_ever else None,
        }
        self.history.append(entry)

    def _print_generation_summary(self, gen: int) -> None:
        fitnesses   = [ind.fitness for ind in self.population]
        best_f      = min(fitnesses)
        mean_f      = np.mean(fitnesses)
        best_ever_f = self.best_ever.fitness if self.best_ever else float("nan")
        mean_sigma  = np.mean([ind.sigma for ind in self.population])

        print(
            f"[ES] Gen {gen:3d}  "
            f"best={best_f:.4f}  mean={mean_f:.4f}  "
            f"best_ever={best_ever_f:.4f}  "
            f"σ_mean={mean_sigma:.4f}  "
            f"stag={self.stagnation}  "
            f"evals={self.n_evals}"
        )


# =============================================================================
# EVALUATE FUNCTION WRAPPER
# =============================================================================

def make_evaluate_fn(
    df_stock:              Any,
    bundle:                Any,
    fixed_norm_constants:  dict,
    config_dict:           dict,
    sample_id_offset:      int = 0,
) -> Callable[[dict], tuple[float, dict]]:
    """
    Wrap evaluate_design_candidate() into the (params → fitness, result) signature
    expected by EvolutionStrategy.

    The wrapper:
    - Assigns a unique sample_id per call (offset + call counter) so geometry
      outputs don't overwrite each other if export_dir is set.
    - Returns (penalty_fitness, result_dict) on pipeline failure rather than
      raising — the ES handles failed evaluations gracefully.

    Usage:
        evaluate_fn = make_evaluate_fn(
            df_stock             = df_input_stock,
            bundle               = SURROGATE_BUNDLE,
            fixed_norm_constants = FIXED_NORMALIZATION_CONSTANTS,
            config_dict          = GA_CONFIG,
        )
        es = EvolutionStrategy(
            search_space = optimizer_search_space,
            evaluate_fn  = evaluate_fn,
            config       = ESConfig(mu=10, lam=20, n_generations=50),
        )
    """
    call_counter = [0]
    penalty      = float(config_dict.get("penalty_fitness", 1e6))

    def _evaluate(design_params: dict) -> tuple[float, dict]:
        call_counter[0] += 1
        sid = sample_id_offset + call_counter[0]

        result = evaluate_design_candidate(
            design_params        = design_params,
            df_stock             = df_stock,
            bundle               = bundle,
            fixed_norm_constants = fixed_norm_constants,
            config_dict          = config_dict,
            sample_id            = sid,
            verbose              = False,
        )

        fitness = float(result.get("fitness", penalty))
        return fitness, result

    return _evaluate


# =============================================================================
# CONVERGENCE PLOT HELPER
# =============================================================================

def plot_convergence(history: list[dict], config: ESConfig = None) -> None:
    """
    Plot best / mean / worst fitness and mean σ over generations.
    Call after es.run() with the returned history list.
    """
    import matplotlib.pyplot as plt
    try:
        from config import PLOT_COLORS as C, PLOT_STYLE as S
    except ImportError:
        C = {"primary": "#61788C", "accent": "#F2994B",
             "danger": "#D9653B", "secondary": "#9CA5A6",
             "neutral": "#D7D9D9", "black": "#000000"}
        S = {"figsize_medium": (12, 7), "dpi": 100, "grid_alpha": 0.3,
             "line_width": 2.0}

    gens      = [h["generation"]    for h in history]
    best      = [h["best_fitness"]  for h in history]
    mean      = [h["mean_fitness"]  for h in history]
    best_ever = [h["best_ever"]     for h in history]
    sigmas    = [h["mean_sigma"]    for h in history]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=S["figsize_medium"])
    fig.suptitle("Evolution Strategy — Convergence", fontweight="bold", fontsize=13)

    ax1.plot(gens, best_ever, color=C["primary"],   lw=S["line_width"],
             label="Best ever")
    ax1.plot(gens, best,      color=C["accent"],    lw=S["line_width"],
             linestyle="--", label="Gen best")
    ax1.plot(gens, mean,      color=C["secondary"], lw=S["line_width"],
             linestyle=":",  label="Gen mean")
    ax1.set_xlabel("Generation")
    ax1.set_ylabel("Fitness (lower = better)")
    ax1.set_title("Fitness Convergence")
    ax1.legend()
    ax1.grid(True, alpha=S["grid_alpha"])
    ax1.spines["top"].set_visible(False)
    ax1.spines["right"].set_visible(False)

    ax2.plot(gens, sigmas, color=C["primary"], lw=S["line_width"])
    ax2.set_xlabel("Generation")
    ax2.set_ylabel("Mean σ (step size)")
    ax2.set_title("Self-Adaptive Step Size")
    ax2.grid(True, alpha=S["grid_alpha"])
    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.show()
    return fig

# SETUP

In [ ]:
# =============================================================================
# c30_ga_setup.py — GA Session Initialisation
# =============================================================================
#
# Run once per notebook session before the GA loop.
# Loads all dependencies, stock data, search space, and GA configuration.
#
# Produces the following session variables:
#   optimizer_search_space        — dict[str, (float, float)]
#   df_input_stock                — pd.DataFrame
#   GA_CONFIG                     — pipeline-level settings dict
#   FIXED_NORMALIZATION_CONSTANTS — initial normalisation bounds
#   MODEL_PREFIX                  — surrogate model artifact stem
#
# ES algorithm parameters (mu, lam, n_generations, sigma_init etc.) are
# defined in ESConfig in c30_ga_runner.py — not here. GA_CONFIG carries
# pipeline-level settings only (weights, MILP, penalty, structural schedule).

import importlib
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import config
from workflows import c24_stage_geometry             as stage_geometry
from workflows import c25_stage_feasibility          as stage_feas       # stage_feas throughout
from workflows import c26_stage_cost_matrix          as stage_cost
from workflows import c27_stage_MILP                 as stage_milp
from workflows import c28_stage_GNN                  as stage_gnn
from workflows import c29_stage_fitness_score        as stage_fitness
from workflows import c29_stage_normalization_bounds as stage_bounds

# Hot-reload all workflow modules so notebook edits are picked up immediately
for _mod in [stage_geometry, stage_feas, stage_cost,
             stage_milp, stage_gnn, stage_fitness, stage_bounds]:
    importlib.reload(_mod)

# =============================================================================
# REPRODUCIBILITY
# =============================================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# =============================================================================
# FLAGS
# =============================================================================

TESTING = False   # True → loads small stock CSV for fast development runs

# =============================================================================
# SEARCH SPACE
# =============================================================================

json_path = config.DATA_IO_PATH / "search_space.json"
if not json_path.exists():
    raise FileNotFoundError(
        f"search_space.json not found at {json_path}. "
        "Generate it from the geometry stage before running the GA."
    )
with open(json_path, "r", encoding="utf-8") as f:
    optimizer_search_space = json.load(f)

print(f"Search space loaded: {len(optimizer_search_space)} variables")

# =============================================================================
# STOCK DATA
# =============================================================================

stock_file = (
    config.TIMBER_STOCK_PATH / "complete_timber_small.csv"
    if TESTING
    else config.TIMBER_STOCK_PATH / "complete_timber.csv"
)
if not stock_file.exists():
    raise FileNotFoundError(f"Stock CSV not found: {stock_file}")

# Try common delimiter/encoding combinations — handles GH exports with ; sep
_df_input_stock = None
for _opts in [
    {"sep": ",",  "encoding": "utf-8"},
    {"sep": ";",  "encoding": "utf-8"},
    {"sep": ",",  "encoding": "latin1"},
    {"sep": ";",  "encoding": "latin1"},
]:
    try:
        _df = pd.read_csv(stock_file, **_opts)
        if _df.shape[1] > 1:
            _df_input_stock = _df
            print(f"Stock loaded: {len(_df)} elements  "
                  f"(sep='{_opts['sep']}', encoding='{_opts['encoding']}')")
            break
    except Exception:
        pass

if _df_input_stock is None:
    raise ValueError(
        f"Could not parse stock CSV at {stock_file}. "
        "Tried , and ; delimiters with utf-8 and latin1 encoding."
    )

df_input_stock = _df_input_stock
df_input_stock.columns = df_input_stock.columns.str.strip()

# =============================================================================
# SURROGATE MODEL PREFIX
# =============================================================================

MODEL_PREFIX = "ID20260509_225351_LR3e-04_EP100_BS32_FA0.50_ROC0.887"

# =============================================================================
# GA CONFIGURATION
#
# Pipeline-level settings only — weights, MILP, penalty, bounds.
# ES algorithm parameters live in ESConfig in c30_ga_runner.py.
# =============================================================================

GA_CONFIG = {
    # Fitness weights — omega_1: cost, omega_2: reuse rate, omega_3: waste
    # omega_4 (structural penalty) is scheduled dynamically — see w_structural_*
    "fitness_weights": {
        "omega_1": 1.0,
        "omega_2": 1.0,
        "omega_3": 1.0,
    },

    # MILP — how many times a new-stock element may be assigned across slots
    "new_stock_max_uses": 100,

    # Penalty fitness assigned to infeasible or failed pipeline evaluations
    "penalty_fitness": 1e6,

    # One-time normalisation bounds computed once at GA startup
    "use_one_time_bounds":   True,
    "bounds_probe_attempts": 8,

    # Structural penalty curriculum — omega_4 ramps linearly over generations
    # Early generations tolerate structural infeasibility; later ones penalise it
    "w_structural_start": 0.2,
    "w_structural_end":   0.8,
}

# =============================================================================
# INITIAL NORMALISATION CONSTANTS
# Overwritten by the one-time bounds solve in c30_ga_evaluator.py
# =============================================================================

# Placeholder — overwritten by one-time bounds solve in c30_ga_evaluator.py
# Do not use this value directly; always run the evaluator cell first.
FIXED_NORMALIZATION_CONSTANTS = stage_fitness.get_default_normalization_constants()

# =============================================================================
# SUMMARY
# =============================================================================

print(f"\n{'='*55}")
print("GA SESSION READY")
print(f"{'='*55}")
print(f"  Search space variables:  {len(optimizer_search_space)}")
print(f"  Stock elements:          {len(df_input_stock)}")
print(f"  Model prefix:            {MODEL_PREFIX}")
print(f"  Fitness weights:         "
      f"ω1={GA_CONFIG['fitness_weights']['omega_1']}  "
      f"ω2={GA_CONFIG['fitness_weights']['omega_2']}  "
      f"ω3={GA_CONFIG['fitness_weights']['omega_3']}")
print(f"  Structural schedule:     "
      f"ω4 {GA_CONFIG['w_structural_start']} → {GA_CONFIG['w_structural_end']}")
print(f"  New stock max uses:      {GA_CONFIG['new_stock_max_uses']}")
print(f"  Fixed normalization:     {FIXED_NORMALIZATION_CONSTANTS}")
print(f"  Penalty fitness:         {GA_CONFIG['penalty_fitness']:.0e}")
print(f"  TESTING mode:            {TESTING}")
print(f"{'='*55}\n")

display(df_input_stock.head(3))

Search space loaded: 73 variables
Stock loaded: 506 elements  (sep=';', encoding='utf-8')

GA SESSION READY
  Search space variables:  73
  Stock elements:          506
  Model prefix:            ID20260509_225351_LR3e-04_EP100_BS32_FA0.50_ROC0.887
  Fitness weights:         ω1=1.0  ω2=1.0  ω3=1.0
  Structural schedule:     ω4 0.2 → 0.8
  New stock max uses:      100
  Fixed normalization:     {'C_max': 8.0, 'R_max': 100.0, 'W_max': 0.4}
  Penalty fitness:         1e+06
  TESTING mode:            False



,Member_ID,State,Length,Depth,Width,f_mk,f_tk,E_modulus_eff,E_modulus_005,f_vk,f_c0k,k_density,mean_density,Origin_Country,Transport_Dist,EmissionFactor
0,NS_00000,0,1450.0,100.0,38.0,24.0,14.0,11000.0,7400.0,2.5,21.0,350.0,420.0,Germany,301.53,0.1796
1,NS_00001,0,1450.0,100.0,50.0,24.0,14.0,11000.0,7400.0,2.5,21.0,350.0,420.0,Netherlands,49.24,0.1756
2,NS_00002,0,1450.0,100.0,63.0,24.0,14.0,11000.0,7400.0,2.5,21.0,350.0,420.0,Netherlands,45.39,0.1735


# EVALUATION

## 24_ Geometry

In [3]:
# =============================================================================
# c30_ga_geometry_check.py — One-time Geometry Sanity Check
# =============================================================================
#
# Optional: run once before the GA to verify the geometry stage produces
# the expected topology (39 nodes, 120 members) and that the output columns
# match what downstream stages expect.
#
# Set RUN_GEOMETRY_CHECK = True to execute; False to skip silently.
# Keep False during production GA runs — this adds ~5-30s unnecessarily.

RUN_GEOMETRY_CHECK = True

EXPECTED_NODES   = 39
EXPECTED_MEMBERS = 120

if RUN_GEOMETRY_CHECK:
    geometry_out = stage_geometry.run_random_geometry_stage(
        optimizer_search_space=optimizer_search_space,
        sample_id=0,
    )

    df_vertices_example         = geometry_out["df_vertices"]
    df_edges_example            = geometry_out["df_edges"]
    df_geometry_overview_example = geometry_out["df_geometry_overview"]

    n_nodes   = len(df_vertices_example)
    n_members = len(df_edges_example)

    print(f"Geometry check: {n_nodes} nodes, {n_members} members")

    if n_nodes != EXPECTED_NODES:
        print(f"  WARNING: expected {EXPECTED_NODES} nodes, got {n_nodes}")
    else:
        print(f"  Nodes:   {n_nodes} / {EXPECTED_NODES} ✓")

    if n_members != EXPECTED_MEMBERS:
        print(f"  WARNING: expected {EXPECTED_MEMBERS} members, got {n_members}")
    else:
        print(f"  Members: {n_members} / {EXPECTED_MEMBERS} ✓")

    display(df_geometry_overview_example[["edge_id", "V1", "V2", "length_m"]].head(5))

else:
    print("Geometry check skipped (RUN_GEOMETRY_CHECK = False).")

Geometry check: 39 nodes, 120 members
  Nodes:   39 / 39 ✓
  Members: 120 / 120 ✓


,edge_id,V1,V2,length_m
0,e0,0,1,3.750
1,e1,0,6,2.625
2,e2,1,2,1.125
3,e3,1,7,2.704
4,e4,2,3,3.000


## 24-27 Iteration

In [ ]:
# =============================================================================
# c30_ga_evaluator_v3.py — Design Evaluator + One-Time Bounds
# =============================================================================
#
# Provides:
#   evaluate_design_candidate()                — full pipeline evaluation
#   _compute_one_time_normalization_constants() — derive C_max/R_max/W_max
#                                                 once before the GA loop
#
# Pipeline order (derived from notebook workflow):
#   geometry → feasibility (slots + forces) → cost matrix → MILP
#           → GNN (on MILP assignment) → normalization bounds → fitness
#
# Requires from c30_ga_setup.py (notebook namespace):
#   optimizer_search_space, df_input_stock, GA_CONFIG, MODEL_PREFIX
#   stage_geometry, stage_feas, stage_cost, stage_milp, stage_fitness,
#   stage_bounds
#
# GNN stage: c28_stage_GNN via c21_surrogate_io (loaded per evaluation
#   via MODEL_PREFIX — matches existing notebook workflow).
#
# Changes vs v2:
#   - stage_feas.build_cost_filter() used correctly — returns
#     (df_slots, feasibility_mask, member_forces, stats)
#   - stage_cost.build_cost_matrix() called with feasibility_mask
#     (not df_utilization_matrix)
#   - GNN called after MILP on real milp_assignment (not dummy zeros)
#   - omega_4 added to weight_config for structural penalty
#   - structural_infeasibility = 1 - feasibility_score passed to fitness
#   - w_structural curriculum scheduling: ramps from w_start to w_end
#     over generations — passed in via config_dict or generation args
#   - normalization bounds derived per evaluation from the same MILP run
#     (not a separate one-time probe)
#   - model_bundle loaded per evaluation via MODEL_PREFIX (matches workflow)

import warnings
import numpy as np
import pandas as pd

import c28_stage_GNN       as stage_gnn
from c21_surrogate_io import load_surrogate_bundle


# =============================================================================
# INTERNAL HELPERS
# =============================================================================

def _resolve_weight_config(
    config_dict: dict,
    w_structural: float = 0.3,
) -> dict:
    """
    Extract omega_1/2/3/4 fitness weights from GA_CONFIG.
    omega_4 is the structural penalty weight — pass current w_structural
    from the curriculum scheduler.
    """
    weights = config_dict.get("fitness_weights", {})
    if isinstance(weights, dict):
        return {
            "omega_1": float(weights.get("omega_1", 1.0)),
            "omega_2": float(weights.get("omega_2", 1.0)),
            "omega_3": float(weights.get("omega_3", 1.0)),
            "omega_4": float(w_structural),
        }

    # Legacy weight_strategy string fallback
    strategy = str(config_dict.get("weight_strategy", "balanced")).strip().lower()
    base = {
        "cost-dominant":  {"omega_1": 1.0, "omega_2": 0.6, "omega_3": 0.8},
        "reuse-dominant": {"omega_1": 0.8, "omega_2": 1.0, "omega_3": 0.8},
        "waste-dominant": {"omega_1": 0.8, "omega_2": 0.6, "omega_3": 1.0},
    }.get(strategy, {"omega_1": 1.0, "omega_2": 1.0, "omega_3": 1.0})
    base["omega_4"] = float(w_structural)
    return base


def _resolve_w_structural(
    config_dict:     dict,
    generation:      int   = 0,
    max_generations: int   = 1,
) -> float:
    """
    Curriculum scheduler for structural penalty weight omega_4.

    Ramps linearly from w_start to w_end over the course of the GA run.
    Early generations tolerate structural infeasibility more (low omega_4),
    later generations penalise it heavily (high omega_4).

    Configure via GA_CONFIG:
        "w_structural_start": 0.2
        "w_structural_end":   0.8
    """
    w_start = float(config_dict.get("w_structural_start", 0.2))
    w_end   = float(config_dict.get("w_structural_end",   0.8))
    if max_generations <= 1:
        return w_end
    t = min(generation / max_generations, 1.0)
    return w_start + (w_end - w_start) * t


def _normalize_bounds_constants(constants: dict) -> dict:
    """Validate and return C_max / R_max / W_max as floats."""
    out = {
        "C_max": float(constants.get("C_max", np.nan)),
        "R_max": float(constants.get("R_max", np.nan)),
        "W_max": float(constants.get("W_max", np.nan)),
    }
    if not all(np.isfinite(list(out.values()))):
        raise ValueError(
            f"Normalization constants contain non-finite values: {out}"
        )
    if any(v <= 0.0 for v in out.values()):
        raise ValueError(
            f"Normalization constants must be strictly positive: {out}"
        )
    return out


# =============================================================================
# ONE-TIME NORMALISATION BOUNDS
# =============================================================================

def _compute_one_time_normalization_constants(
    search_space: dict,
    df_stock,
    config_dict:  dict,
) -> tuple[dict, dict]:
    """
    Derive C_max / R_max / W_max by running the full pipeline on random
    probe designs once before the GA loop starts.

    Returns (normalization_constants, source_info).
    """
    defaults = stage_fitness.get_default_normalization_constants()

    if not bool(config_dict.get("use_one_time_bounds", True)):
        return defaults, {
            "source": "defaults",
            "reason": "one-time bounds disabled in GA_CONFIG",
        }

    attempts           = max(int(config_dict.get("bounds_probe_attempts", 8)), 1)
    new_stock_max_uses = config_dict.get("new_stock_max_uses", 1)

    for attempt_idx in range(attempts):
        try:
            probe_design = stage_geometry.sample_random_design(search_space)

            geo_out = stage_geometry.run_geometry_from_design(
                design_params = probe_design,
                sample_id     = 10_000 + attempt_idx,
            )

            verts           = geo_out["df_vertices"]
            node_positions  = verts[["x", "y", "z"]].values
            support_nodes   = verts[verts["attribute"] == "support"]["v_idx"].tolist()
            load_nodes      = verts[verts["attribute"] == "load"]["v_idx"].tolist()

            df_slots, feasibility_mask, member_forces, _ = stage_feas.build_cost_filter(
                node_positions = node_positions,
                edges_df       = geo_out["df_edges"],
                stock_df       = df_stock,
                support_nodes  = support_nodes,
                load_nodes     = load_nodes,
            )

            cost_matrix, stock_prepared, logs = stage_cost.build_cost_matrix(
                df_slots         = df_slots,
                df_input_stock   = df_stock,
                feasibility_mask = feasibility_mask,
            )

            milp_out = stage_milp.run_milp_stage(
                cost_matrix    = cost_matrix,
                enriched_stock = stock_prepared,
                df_slots       = df_slots,
                reclaimed_marker          = "RS",
                new_marker                = "NS",
                new_stock_max_uses        = (
                    None if new_stock_max_uses is None
                    else int(new_stock_max_uses)
                ),
                solver_msg                = False,
                raise_on_infeasible_slots = False,
            )

            if milp_out["status"] != "Optimal":
                print(f"  [bounds probe {attempt_idx+1}] "
                      f"MILP {milp_out['status']} — retrying")
                continue

            bounds_out = stage_bounds.run_normalization_bounds_stage(
                cost_matrix    = cost_matrix,
                df_logs        = logs,
                enriched_stock = stock_prepared,
                df_slots       = df_slots,
                reclaimed_marker   = "RS",
                new_marker         = "NS",
                new_stock_max_uses = (
                    None if new_stock_max_uses is None
                    else int(new_stock_max_uses)
                ),
                solver_msg    = False,
                print_summary = False,
            )

            if str(bounds_out.get("status", "")).lower() not in {"optimal", "partial"}:
                print(f"  [bounds probe {attempt_idx+1}] bounds status "
                      f"'{bounds_out.get('status')}' — retrying")
                continue

            normalized = _normalize_bounds_constants(
                bounds_out["normalization_constants"]
            )
            print(f"  [bounds probe {attempt_idx+1}] success → {normalized}")
            return normalized, {
                "source":  "one-time-bounds",
                "status":  bounds_out.get("status"),
                "attempt": attempt_idx + 1,
            }

        except Exception as exc:
            print(f"  [bounds probe {attempt_idx+1}] exception: {exc} — retrying")
            continue

    warnings.warn(
        f"Could not derive valid one-time bounds after {attempts} attempts. "
        "Falling back to default normalization constants.",
        stacklevel=2,
    )
    return defaults, {
        "source": "defaults",
        "reason": f"all {attempts} probe attempts failed",
    }


# =============================================================================
# DESIGN EVALUATOR
# =============================================================================

def evaluate_design_candidate(
    design_params:        dict,
    df_stock,
    fixed_norm_constants: dict,
    config_dict:          dict,
    generation:           int  = 0,
    max_generations:      int  = 1,
    sample_id:            int  = 0,
    verbose:              bool = False,
) -> dict:
    """
    Evaluate one design candidate through the full pipeline.

    Pipeline:
        geometry → feasibility (slots + forces) → cost matrix
        → MILP → GNN (on MILP assignment) → fitness

    Parameters
    ----------
    design_params        : continuous geometry parameters from ES individual
    df_stock             : full stock DataFrame
    fixed_norm_constants : C_max/R_max/W_max from one-time bounds solve
    config_dict          : GA_CONFIG
    generation           : current generation (for w_structural curriculum)
    max_generations      : total generations (for w_structural curriculum)
    sample_id            : unique ID for this evaluation
    verbose              : print one-line summary on success

    Returns
    -------
    result : dict with keys:
        status             — "OK" | "PENALIZED"
        fitness            — float (lower = better)
        reason             — None on OK, error string on PENALIZED
        milp_status        — MILP solver status string
        total_cost         — float
        reuse_rate         — float
        waste_total        — float
        fitness_result     — full fitness breakdown dict
        gnn_feasibility    — float [0,1] fraction of safe members
        gnn_unsafe_members — list[int] unsafe member IDs
        w_structural       — float omega_4 used this evaluation
        df_vertices        — geometry nodes DataFrame
        df_edges           — geometry edges DataFrame
        df_results         — MILP assignment DataFrame
        design_params      — echo of input
    """
    penalty = float(config_dict["penalty_fitness"])
    result  = {
        "design_params":       design_params,
        "status":              "UNKNOWN",
        "fitness":             penalty,
        "reason":              None,
        "fitness_result":      None,
        "milp_status":         None,
        "total_cost":          float("inf"),
        "reuse_rate":          0.0,
        "waste_total":         float("inf"),
        "gnn_feasibility":     None,
        "gnn_unsafe_members":  None,
        "w_structural":        None,
        "df_vertices":         None,
        "df_edges":            None,
        "df_results":          None,
    }

    try:
        # ---- geometry -------------------------------------------------------
        geo_out     = stage_geometry.run_geometry_from_design(
            design_params = design_params,
            sample_id     = int(sample_id),
        )
        df_vertices = geo_out["df_vertices"]
        df_edges    = geo_out["df_edges"]

        # Derive node roles from geometry output
        verts         = df_vertices.copy()
        verts["v_idx"] = verts["vertex_index"].str.replace("v", "").astype(int)
        verts         = verts.sort_values("v_idx").reset_index(drop=True)
        node_positions = verts[["x", "y", "z"]].values
        support_nodes  = verts[verts["attribute"] == "support"]["v_idx"].tolist()
        load_nodes     = verts[verts["attribute"] == "load"]["v_idx"].tolist()

        # ---- feasibility (slots + member forces) ----------------------------
        df_slots, feasibility_mask, member_forces, _ = stage_feas.build_cost_filter(
            node_positions = node_positions,
            edges_df       = df_edges,
            stock_df       = df_stock,
            support_nodes  = support_nodes,
            load_nodes     = load_nodes,
        )

        # ---- cost matrix ----------------------------------------------------
        cost_matrix, stock_prepared, logs = stage_cost.build_cost_matrix(
            df_slots         = df_slots,
            df_input_stock   = df_stock,
            feasibility_mask = feasibility_mask,
        )

        # ---- MILP -----------------------------------------------------------
        new_stock_max_uses = config_dict.get("new_stock_max_uses", 1)
        milp_out = stage_milp.run_milp_stage(
            cost_matrix    = cost_matrix,
            enriched_stock = stock_prepared,
            df_slots       = df_slots,
            reclaimed_marker          = "RS",
            new_marker                = "NS",
            new_stock_max_uses        = (
                None if new_stock_max_uses is None
                else int(new_stock_max_uses)
            ),
            solver_msg                = False,
            raise_on_infeasible_slots = False,
        )

        result["milp_status"] = milp_out["status"]
        if milp_out["status"] != "Optimal":
            result["status"] = "PENALIZED"
            result["reason"] = f"MILP status: {milp_out['status']}"
            return result

        df_results      = milp_out["df_results"]
        total_cost      = milp_out["total_cost"]
        milp_assignment = milp_out.get("milp_assignment")

        # ---- GNN feasibility (on actual MILP assignment) --------------------
        gnn_feasibility    = 1.0
        gnn_unsafe_members = []

        if milp_assignment is not None and MODEL_PREFIX:
            model_bundle = load_surrogate_bundle(prefix_sm=MODEL_PREFIX)
            gnn_out = stage_gnn.run_gnn_stage(
                node_positions  = node_positions,
                milp_assignment = milp_assignment,
                df_input_stock  = df_stock,
                model_bundle    = model_bundle,
                print_summary   = False,
            )
            gnn_feasibility    = float(gnn_out["feasibility_score"])
            gnn_unsafe_members = gnn_out["unsafe_member_ids"]
        else:
            if not MODEL_PREFIX:
                warnings.warn(
                    "MODEL_PREFIX not set — GNN stage skipped. "
                    "Set MODEL_PREFIX in c30_ga_setup.py.",
                    stacklevel=2,
                )
            if milp_assignment is None:
                warnings.warn(
                    "milp_assignment not found in milp_out — GNN stage skipped.",
                    stacklevel=2,
                )

        result["gnn_feasibility"]    = gnn_feasibility
        result["gnn_unsafe_members"] = gnn_unsafe_members

        # ---- w_structural curriculum ----------------------------------------
        w_structural = _resolve_w_structural(
            config_dict     = config_dict,
            generation      = generation,
            max_generations = max_generations,
        )
        result["w_structural"] = w_structural

        # ---- fitness --------------------------------------------------------
        weight_config = _resolve_weight_config(config_dict, w_structural)

        fitness_out = stage_fitness.run_fitness_stage(
            df_results              = df_results,
            enriched_stock          = stock_prepared,
            df_slots                = df_slots,
            total_cost              = total_cost,
            weight_config           = weight_config,
            normalization_constants = fixed_norm_constants,
            structural_infeasibility = 1.0 - gnn_feasibility,
            derive_normalization_constants = False,
            run_sanity_checks       = False,
            print_breakdown         = False,
        )

        fitness_result = fitness_out["fitness_result"]
        result.update({
            "status":         "OK",
            "fitness":        float(fitness_result["fitness"]),
            "reason":         None,
            "fitness_result": fitness_result,
            "total_cost":     float(fitness_result.get("cost_raw", total_cost)),
            "reuse_rate":     float(fitness_result.get("reuse_rate", 0.0)),
            "waste_total":    float(fitness_result.get("waste_total", 0.0)),
            "df_vertices":    df_vertices,
            "df_edges":       df_edges,
            "df_results":     df_results,
        })

        if verbose:
            print(
                f"  OK | fitness={result['fitness']:.4f} | "
                f"cost={result['total_cost']:.2f} | "
                f"reuse={result['reuse_rate']:.1%} | "
                f"gnn={gnn_feasibility:.2%} | "
                f"ω4={w_structural:.2f}"
            )

    except Exception as exc:
        result["status"] = "PENALIZED"
        result["reason"] = str(exc)

    return result


# =============================================================================
# ONE-TIME BOUNDS SOLVE + SMOKE TEST
# =============================================================================

FIXED_NORMALIZATION_CONSTANTS, BOUNDS_SOURCE_INFO = (
    _compute_one_time_normalization_constants(
        search_space = optimizer_search_space,
        df_stock     = df_input_stock,
        config_dict  = GA_CONFIG,
    )
)
print(f"\nNormalisation constants: {FIXED_NORMALIZATION_CONSTANTS}")
print(f"Source: {BOUNDS_SOURCE_INFO}")

# Smoke test — one full evaluation to catch wiring issues before GA
print("\nRunning evaluator smoke test...")
_test_design = stage_geometry.sample_random_design(optimizer_search_space)
_test_eval   = evaluate_design_candidate(
    design_params        = _test_design,
    df_stock             = df_input_stock,
    fixed_norm_constants = FIXED_NORMALIZATION_CONSTANTS,
    config_dict          = GA_CONFIG,
    generation           = 0,
    max_generations      = 1,
    sample_id            = 99_999,
    verbose              = True,
)
print(f"Smoke test status: {_test_eval['status']}")
if _test_eval["reason"] is not None:
    print(f"Reason: {_test_eval['reason']}")
print("Evaluator ready.")

# OUTPUT

In [ ]:
# Run custom GA loop (balanced defaults)
population_size = int(GA_CONFIG["population_size"])
generations = int(GA_CONFIG["generations"])
elite_count = int(GA_CONFIG["elite_count"])
tournament_k = int(GA_CONFIG["tournament_k"])
crossover_rate = float(GA_CONFIG["crossover_rate"])
mutation_rate = float(GA_CONFIG["mutation_rate"])
mutation_scale = float(GA_CONFIG["mutation_scale"])

population = initialize_population(optimizer_search_space, population_size)
ga_history_rows = []
best_overall = None

for gen in range(generations):
    evals = []
    fitnesses = []

    for ind_idx, individual in enumerate(population):
        ev = evaluate_design_candidate(
            design_params=individual,
            df_stock=df_input_stock,
            bundle=SURROGATE_BUNDLE,
            fixed_norm_constants=FIXED_NORMALIZATION_CONSTANTS,
            config_dict=GA_CONFIG,
            model_prefix=MODEL_PREFIX,
            sample_id=ind_idx,
            verbose=False,
        )
        evals.append(ev)
        fitnesses.append(float(ev["fitness"]))

        ga_history_rows.append(
            {
                "generation": gen,
                "individual": ind_idx,
                "fitness": float(ev["fitness"]),
                "status": ev["status"],
                "reason": ev["reason"],
                "milp_status": ev.get("milp_status"),
                "total_cost": float(ev.get("total_cost", float("inf"))),
                "reuse_rate": float(ev.get("reuse_rate", 0.0)),
                "waste_total": float(ev.get("waste_total", float("inf"))),
                "design_params_json": json.dumps(ev["design_params"]),
            }
        )

    order = np.argsort(fitnesses)
    ranked = [evals[i] for i in order]
    best_gen = ranked[0]

    if best_overall is None or float(best_gen["fitness"]) < float(best_overall["fitness"]):
        best_overall = best_gen

    finite_fit = [f for f in fitnesses if np.isfinite(f)]
    mean_fit = float(np.mean(finite_fit)) if finite_fit else float("inf")
    feasible_count = int(sum(1 for r in evals if r["status"] == "OK"))
    print(
        f"Gen {gen + 1:02d}/{generations} | "
        f"best={float(best_gen['fitness']):.4f} | "
        f"mean={mean_fit:.4f} | "
        f"feasible={feasible_count}/{len(evals)}"
    )

    # Elitism + reproduction
    next_population = [ranked[i]["design_params"].copy() for i in range(min(elite_count, len(ranked)))]

    while len(next_population) < population_size:
        p1 = tournament_selection(population, fitnesses, k=tournament_k)
        p2 = tournament_selection(population, fitnesses, k=tournament_k)

        if random.random() < crossover_rate:
            child = crossover(p1, p2, optimizer_search_space)
        else:
            child = p1.copy()

        child = mutate(
            child,
            optimizer_search_space,
            mutation_rate=mutation_rate,
            mutation_scale=mutation_scale,
        )
        next_population.append(child)

    population = next_population

ga_history_df = pd.DataFrame(ga_history_rows)
ga_ranked_df = ga_history_df.sort_values(by=["fitness", "generation", "individual"]).reset_index(drop=True)

print("\nGA finished.")
print(f"Best fitness: {float(best_overall['fitness']):.4f}")
display(ga_ranked_df.head(10))

## GA Analysis & Diagnostics

**Purpose:** Deep inspection of GA behavior to catch anomalies, validate convergence, and understand design patterns.

**Included Analyses:**
- **Convergence**: Best/mean/worst fitness per generation (detects stalling or divergence)
- **Feasibility**: % viable solutions per generation (detects infeasibility waves)
- **Parameter Correlation**: Which design variables correlate with low fitness?
- **Elite Performance**: How well did selected individuals perform?
- **Fitness Distribution**: Histogram to spot bimodal/weird distributions
- **Top Designs**: Ranked table with all key metrics
- **Anomaly Detection**: Flags unusual fitness spikes, all-infeasible generations, parameter outliers
- **Text Summary**: Key findings + warnings about potential issues


In [ ]:
import json
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# Load centralized color theme
from config import PLOT_COLORS, PLOT_STYLE

# -----------------------------------------------------------------------------
# Analysis export directory (03_ga_data/<run_id>)
# -----------------------------------------------------------------------------
def _fmt_float_tag(x, ndigits=2):
    return f"{float(x):.{ndigits}f}".replace(".", "p")

ga_tag = (
    f"ID{datetime.now():%Y%m%d_%H%M%S}"
    f"_PS{int(GA_CONFIG['population_size'])}"
    f"_G{int(GA_CONFIG['generations'])}"
    f"_E{int(GA_CONFIG['elite_count'])}"
    f"_T{int(GA_CONFIG['tournament_k'])}"
    f"_CR{_fmt_float_tag(GA_CONFIG['crossover_rate'])}"
    f"_MR{_fmt_float_tag(GA_CONFIG['mutation_rate'])}"
    f"_MS{_fmt_float_tag(GA_CONFIG['mutation_scale'])}"
    f"_UT{_fmt_float_tag(GA_CONFIG['utilization_threshold'])}"
    f"_SFM{str(GA_CONFIG.get('surrogate_edge_feature_mode', 'na'))}"
)

GA_ANALYSIS_RUN_ID = ga_tag
GA_ANALYSIS_DIR = config.GA_DATA_PATH / GA_ANALYSIS_RUN_ID
GA_ANALYSIS_FIG_DIR = GA_ANALYSIS_DIR / "figures"
GA_ANALYSIS_TABLE_DIR = GA_ANALYSIS_DIR / "tables"
GA_ANALYSIS_META_DIR = GA_ANALYSIS_DIR / "meta"
GA_ANALYSIS_FIG_DIR.mkdir(parents=True, exist_ok=True)
GA_ANALYSIS_TABLE_DIR.mkdir(parents=True, exist_ok=True)
GA_ANALYSIS_META_DIR.mkdir(parents=True, exist_ok=True)

print(f"Analysis export run id: {GA_ANALYSIS_RUN_ID}")
print(f"Analysis export folder: {GA_ANALYSIS_DIR}")

# Split feasible vs all evaluations for robust diagnostics
ga_ok_df = ga_history_df[ga_history_df["status"] == "OK"].copy()
has_ok = len(ga_ok_df) > 0

# ============================================================================
# 1. CONVERGENCE ANALYSIS
# ============================================================================
gen_stats = ga_history_df.groupby("generation").agg({
    "fitness": ["min", "max", "mean", "std"],
    "status": lambda x: (x == "OK").sum() / len(x) * 100,
    "total_cost": ["min", "max", "mean"],
}).round(4)
gen_stats.columns = ["best_fit", "worst_fit", "mean_fit", "std_fit", "feasible_pct", "best_cost", "worst_cost", "mean_cost"]

fig, axes = plt.subplots(2, 2, figsize=PLOT_STYLE["figsize_large"])

# Convergence curve
ax = axes[0, 0]
ax.plot(gen_stats.index, gen_stats["best_fit"],
        color=PLOT_COLORS["primary"], linewidth=PLOT_STYLE["line_width"],
        label="Best", marker="o", markersize=PLOT_STYLE["marker_size"])
ax.plot(gen_stats.index, gen_stats["mean_fit"],
        color=PLOT_COLORS["secondary"], linewidth=PLOT_STYLE["line_width"],
        label="Mean", marker="s", markersize=PLOT_STYLE["marker_size"], linestyle="--")
ax.plot(gen_stats.index, gen_stats["worst_fit"],
        color=PLOT_COLORS["danger"], linewidth=1, alpha=0.7,
        label="Worst", marker="^", markersize=4)
ax.fill_between(gen_stats.index, gen_stats["best_fit"], gen_stats["worst_fit"],
                alpha=0.15, color=PLOT_COLORS["neutral"])
ax.set_xlabel("Generation", fontsize=11)
ax.set_ylabel("Fitness (lower = better)", fontsize=11)
ax.set_title("GA Convergence: Fitness Trajectory", fontsize=12, fontweight="bold")
ax.legend(loc="best")
ax.grid(True, alpha=PLOT_STYLE["grid_alpha"], color=PLOT_COLORS["neutral"])

# Feasibility rate
ax = axes[0, 1]
ax.bar(gen_stats.index, gen_stats["feasible_pct"],
       color=PLOT_COLORS["primary"], alpha=0.7, edgecolor=PLOT_COLORS["black"], linewidth=1.5)
ax.axhline(y=100, color=PLOT_COLORS["accent"], linestyle="--", linewidth=2, label="100% Target")
ax.set_xlabel("Generation", fontsize=11)
ax.set_ylabel("Feasible Solutions (%)", fontsize=11)
ax.set_title("Solution Feasibility Over Time", fontsize=12, fontweight="bold")
ax.set_ylim([0, 105])
ax.legend()
ax.grid(True, alpha=PLOT_STYLE["grid_alpha"], axis="y", color=PLOT_COLORS["neutral"])

# Cost trend
ax = axes[1, 0]
ax.plot(gen_stats.index, gen_stats["best_cost"],
        color=PLOT_COLORS["primary"], linewidth=PLOT_STYLE["line_width"],
        label="Best Cost", marker="o", markersize=PLOT_STYLE["marker_size"])
ax.plot(gen_stats.index, gen_stats["mean_cost"],
        color=PLOT_COLORS["secondary"], linewidth=PLOT_STYLE["line_width"],
        label="Mean Cost", marker="s", markersize=PLOT_STYLE["marker_size"], linestyle="--")
ax.set_xlabel("Generation", fontsize=11)
ax.set_ylabel("Total Carbon (kg CO2e)", fontsize=11)
ax.set_title("Carbon Cost Progression", fontsize=12, fontweight="bold")
ax.legend()
ax.grid(True, alpha=PLOT_STYLE["grid_alpha"], color=PLOT_COLORS["neutral"])

# Fitness distribution (last generation)
ax = axes[1, 1]
last_gen_fitness = ga_history_df[ga_history_df["generation"] == ga_history_df["generation"].max()]["fitness"]
ax.hist(last_gen_fitness, bins=10, color=PLOT_COLORS["primary"], alpha=0.7,
        edgecolor=PLOT_COLORS["black"], linewidth=1.5)
ax.axvline(last_gen_fitness.mean(), color=PLOT_COLORS["danger"], linestyle="--",
           linewidth=2, label=f"Mean: {last_gen_fitness.mean():.4f}")
ax.axvline(last_gen_fitness.min(), color=PLOT_COLORS["accent"], linestyle="--",
           linewidth=2, label=f"Best: {last_gen_fitness.min():.4f}")
ax.set_xlabel("Fitness", fontsize=11)
ax.set_ylabel("Count", fontsize=11)
ax.set_title(f"Final Generation Fitness Distribution (Gen {ga_history_df['generation'].max()})",
             fontsize=12, fontweight="bold")
ax.legend()
ax.grid(True, alpha=PLOT_STYLE["grid_alpha"], axis="y", color=PLOT_COLORS["neutral"])

fig.savefig(GA_ANALYSIS_FIG_DIR / "01_convergence_dashboard.png", dpi=300, bbox_inches="tight")
plt.tight_layout()
plt.show()

print("=" * 80)
print("CONVERGENCE STATISTICS")
print("=" * 80)
print(gen_stats.to_string())

# ============================================================================
# 2. ANOMALY DETECTION
# ============================================================================
print("\n" + "=" * 80)
print("ANOMALY DETECTION & WARNINGS")
print("=" * 80)

warnings_list = []

# Check for abrupt regressions in best fitness (delta-based, robust to negative values)
spike_threshold = 0.10
for gen in range(1, len(gen_stats)):
    prev_best = float(gen_stats.iloc[gen - 1]["best_fit"])
    curr_best = float(gen_stats.iloc[gen]["best_fit"])
    if (curr_best - prev_best) > spike_threshold:
        warnings_list.append(
            f"GEN {gen}: Best-fitness regression detected (delta={curr_best - prev_best:+.4f})"
        )

# Check for low-feasibility generations
all_infeasible_gens = gen_stats[gen_stats["feasible_pct"] < 50].index.tolist()
if all_infeasible_gens:
    warnings_list.append(f"INFEASIBILITY: Generations {all_infeasible_gens} had <50% feasible solutions")

# Check for stagnation (no improvement for multiple generations)
improvement_threshold = 0.0001
stagnant_gens = 0
for gen in range(1, len(gen_stats)):
    improvement = gen_stats.iloc[gen - 1]["best_fit"] - gen_stats.iloc[gen]["best_fit"]
    if improvement < improvement_threshold:
        stagnant_gens += 1
    else:
        stagnant_gens = 0
    if stagnant_gens >= 5:
        warnings_list.append(
            f"STAGNATION: No improvement in best fitness for 5+ generations (Gen {gen - 4} to {gen})"
        )
        break

# Check for high variance in last gen (population diversity)
last_gen_std = float(gen_stats.iloc[-1]["std_fit"])
last_gen_mean = float(gen_stats.iloc[-1]["mean_fit"])
cv = last_gen_std / max(abs(last_gen_mean), 1e-12)
if cv > 0.5:
    warnings_list.append(f"HIGH DIVERSITY: Final population has high variance (|CV|={cv:.3f}), may not have converged")

# Check for zero fitness improvement
if gen_stats.iloc[-1]["best_fit"] >= gen_stats.iloc[0]["best_fit"]:
    warnings_list.append("NO IMPROVEMENT: Best fitness did not improve from Gen 0 to final generation")

if not warnings_list:
    print("No anomalies detected. GA ran normally.")
else:
    for warn in warnings_list:
        print(warn)

# ============================================================================
# 3. ELITE PERFORMANCE
# ============================================================================
print("\n" + "=" * 80)
print("ELITE INDIVIDUALS (TOP 10 RANKED DESIGNS)")
print("=" * 80)
top_10 = ga_ranked_df.head(10)[["generation", "individual", "fitness", "status", "milp_status", "total_cost", "reuse_rate", "waste_total"]].copy()
top_10["reuse_rate"] = top_10["reuse_rate"].apply(lambda x: f"{x:.1f}%" if pd.notna(x) else "N/A")
top_10["total_cost"] = top_10["total_cost"].apply(lambda x: f"{x:.2f}" if pd.notna(x) else "N/A")
top_10["waste_total"] = top_10["waste_total"].apply(lambda x: f"{x:.3f}" if pd.notna(x) else "N/A")
top_10["fitness"] = top_10["fitness"].apply(lambda x: f"{x:.6f}")
print(top_10.to_string(index=False))

# ============================================================================
# 4. FEASIBILITY BREAKDOWN
# ============================================================================
print("\n" + "=" * 80)
print("SOLUTION STATUS BREAKDOWN")
print("=" * 80)
status_counts = ga_history_df["status"].value_counts()
print(f"Total individuals evaluated: {len(ga_history_df)}")
for status, count in status_counts.items():
    pct = (count / len(ga_history_df)) * 100
    print(f"  {status}: {count} ({pct:.1f}%)")

# MILP status breakdown (for feasible only)
milp_breakdown = ga_history_df[ga_history_df["status"] == "OK"]["milp_status"].value_counts()
if not milp_breakdown.empty:
    print("\nMILP Solution Status (OK individuals only):")
    for status, count in milp_breakdown.items():
        pct = (count / len(ga_history_df[ga_history_df["status"] == "OK"])) * 100
        print(f"  {status}: {count} ({pct:.1f}%)")

# ============================================================================
# 5. PARAMETER CORRELATION ANALYSIS (feasible designs only)
# ============================================================================
print("\n" + "=" * 80)
print("PARAMETER SENSITIVITY: Correlation with Fitness")
print("=" * 80)

# Extract design params and fitness from feasible rows only
design_param_rows = []
for _, row in ga_ok_df.iterrows():
    try:
        params = json.loads(row["design_params_json"])
        params["fitness"] = row["fitness"]
        design_param_rows.append(params)
    except Exception:
        pass

print(f"Using {len(design_param_rows)} feasible designs for sensitivity analysis.")

correlations_sorted = []
if design_param_rows:
    df_params = pd.DataFrame(design_param_rows)

    # Calculate correlations
    numeric_cols = df_params.select_dtypes(include=[np.number]).columns.tolist()
    if "fitness" in numeric_cols:
        numeric_cols.remove("fitness")

    correlations = {}
    for col in numeric_cols:
        corr = df_params[col].corr(df_params["fitness"])
        if pd.notna(corr):
            correlations[col] = corr

    # Sort by absolute correlation
    correlations_sorted = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)

    print("\nTop parameters correlated with fitness (negative = lower fitness when param increases):")
    for param, corr in correlations_sorted[:8]:
        direction = "up helps" if corr < 0 else "down hurts"
        print(f"  {param:20s}: {corr:+.4f}  ({direction})")

    # Correlation bar chart
    if len(correlations_sorted) >= 3:
        fig_corr, ax = plt.subplots(figsize=PLOT_STYLE["figsize_medium"])
        top_params = [x[0] for x in correlations_sorted[:8]]
        corr_values = [x[1] for x in correlations_sorted[:8]]
        colors = [PLOT_COLORS["accent"] if c < 0 else PLOT_COLORS["danger"] for c in corr_values]
        ax.barh(range(len(top_params)), corr_values, color=colors, alpha=0.7,
                edgecolor=PLOT_COLORS["black"], linewidth=1.5)
        ax.set_yticks(range(len(top_params)))
        ax.set_yticklabels(top_params)
        ax.set_xlabel("Correlation with Fitness (negative = better for low cost)", fontsize=11)
        ax.set_title("Design Parameter Sensitivity", fontsize=12, fontweight="bold")
        ax.axvline(x=0, color=PLOT_COLORS["black"], linewidth=1.0)
        ax.grid(True, alpha=PLOT_STYLE["grid_alpha"], axis="x", color=PLOT_COLORS["neutral"])
        fig_corr.savefig(GA_ANALYSIS_FIG_DIR / "02_parameter_sensitivity.png", dpi=300, bbox_inches="tight")
        plt.tight_layout()
        plt.show()
else:
    print("Could not extract feasible design parameters for correlation analysis.")

# -----------------------------------------------------------------------------
# Save analysis data artifacts
# -----------------------------------------------------------------------------
gen_stats.to_csv(GA_ANALYSIS_TABLE_DIR / "gen_stats.csv")
last_gen_fitness.to_frame(name="fitness").to_csv(GA_ANALYSIS_TABLE_DIR / "last_generation_fitness.csv", index=False)
top_10.to_csv(GA_ANALYSIS_TABLE_DIR / "top_10_ranked_designs.csv", index=False)
status_counts.rename("count").to_csv(GA_ANALYSIS_TABLE_DIR / "status_counts.csv")
if not milp_breakdown.empty:
    milp_breakdown.rename("count").to_csv(GA_ANALYSIS_TABLE_DIR / "milp_status_breakdown_ok.csv")
if design_param_rows:
    pd.DataFrame(design_param_rows).to_csv(GA_ANALYSIS_TABLE_DIR / "design_params_feasible.csv", index=False)
if correlations_sorted:
    pd.DataFrame(correlations_sorted, columns=["parameter", "correlation"]).to_csv(
        GA_ANALYSIS_TABLE_DIR / "parameter_correlations.csv", index=False
    )

analysis_meta = {
    "run_id": GA_ANALYSIS_RUN_ID,
    "analysis_export_dir": str(GA_ANALYSIS_DIR),
    "total_evaluations": int(len(ga_history_df)),
    "feasible_evaluations": int(len(ga_ok_df)),
    "warnings": warnings_list,
}
with open(GA_ANALYSIS_META_DIR / "analysis_meta.json", "w", encoding="utf-8") as f:
    json.dump(analysis_meta, f, indent=2)

print(f"\nAnalysis artifacts exported to: {GA_ANALYSIS_DIR}")
print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)


In [ ]:
# ============================================================================
# EXTENDED DIAGNOSTICS: Multi-Criteria Analysis
# ============================================================================

print("\n" + "=" * 80)
print("MULTI-CRITERIA TRADE-OFF ANALYSIS")
print("=" * 80)

# Restrict ranking to feasible designs only
diagnostic_source = ga_ranked_df[ga_ranked_df["status"] == "OK"].copy()
if diagnostic_source.empty:
    diagnostic_source = ga_ranked_df.copy()
    print("Warning: no feasible designs found, using all designs for diagnostic ranking.")

# Create multi-objective ranking
diagnostic_df = diagnostic_source.head(30).copy()
diagnostic_df["cost_rank"] = diagnostic_df["total_cost"].rank()
diagnostic_df["reuse_rank"] = diagnostic_df["reuse_rate"].rank(ascending=False)
diagnostic_df["waste_rank"] = diagnostic_df["waste_total"].rank()
diagnostic_df["fitness_rank"] = diagnostic_df["fitness"].rank()
diagnostic_df["overall_score"] = (
    diagnostic_df["cost_rank"] +
    diagnostic_df["reuse_rank"] +
    diagnostic_df["waste_rank"]
) / 3

print("\nTop designs by different criteria:")
print("\n1. LOWEST EMBODIED CARBON:")
cost_winners = diagnostic_df.nsmallest(3, "total_cost")[["generation", "individual", "fitness", "total_cost", "reuse_rate", "waste_total"]]
print(cost_winners.to_string(index=False))

print("\n2. HIGHEST REUSE RATE:")
reuse_winners = diagnostic_df.nlargest(3, "reuse_rate")[["generation", "individual", "fitness", "total_cost", "reuse_rate", "waste_total"]]
print(reuse_winners.to_string(index=False))

print("\n3. LOWEST WASTE VOLUME:")
waste_winners = diagnostic_df.nsmallest(3, "waste_total")[["generation", "individual", "fitness", "total_cost", "reuse_rate", "waste_total"]]
print(waste_winners.to_string(index=False))

print("\n4. OVERALL BEST (multi-criteria balanced):")
overall_winners = diagnostic_df.nsmallest(3, "overall_score")[["generation", "individual", "fitness", "total_cost", "reuse_rate", "waste_total", "overall_score"]]
print(overall_winners.to_string(index=False))

# ============================================================================
# GENERATION-BY-GENERATION DETAILED STATS
# ============================================================================
print("\n" + "=" * 80)
print("GENERATION-BY-GENERATION PERFORMANCE DETAIL")
print("=" * 80)

gen_detail = ga_history_df.groupby("generation").agg({
    "fitness": ["count", "min", "mean", "max", lambda x: (x < 1e6).sum()],
    "total_cost": ["min", "mean", "max"],
    "reuse_rate": ["min", "mean", "max"],
}).round(3)

gen_detail.columns = ["n_eval", "best_fit", "mean_fit", "worst_fit", "n_feasible",
                       "best_cost", "mean_cost", "worst_cost",
                       "min_reuse%", "mean_reuse%", "max_reuse%"]

print(gen_detail.to_string())

# ============================================================================
# SEARCH SPACE EXPLORATION CHECK
# ============================================================================
print("\n" + "=" * 80)
print("SEARCH SPACE EXPLORATION ASSESSMENT")
print("=" * 80)

if design_param_rows:
    df_params = pd.DataFrame(design_param_rows)
    numeric_cols = df_params.select_dtypes(include=[np.number]).columns.tolist()
    if "fitness" in numeric_cols:
        numeric_cols.remove("fitness")

    print(f"\nDesign parameters explored across {len(design_param_rows)} feasible individuals:\n")
    for col in numeric_cols[:8]:  # Show top 8 params
        if col in df_params.columns:
            min_val = df_params[col].min()
            max_val = df_params[col].max()

            # Check if we're using full search space
            if col in optimizer_search_space:
                space = optimizer_search_space[col]
                if space["type"] == "continuous":
                    space_min = space["min"]
                    space_max = space["max"]
                    space_range = space_max - space_min
                    explored_pct = ((max_val - min_val) / space_range * 100) if space_range > 0 else 0

                    print(f"{col:20s}: [{min_val:+.4f}, {max_val:+.4f}] (range: {max_val-min_val:+.4f}) - {explored_pct:.1f}% of search space")
                else:
                    unique_vals = df_params[col].nunique()
                    print(f"{col:20s}: {unique_vals} unique values (discrete)")
            else:
                print(f"{col:20s}: [{min_val:+.4f}, {max_val:+.4f}]")
else:
    print("No feasible design-parameter rows available for exploration assessment.")

# ============================================================================
# FAILURE/ERROR ANALYSIS
# ============================================================================
print("\n" + "=" * 80)
print("FAILURE ANALYSIS")
print("=" * 80)

penalized = ga_history_df[ga_history_df["status"] == "PENALIZED"]
if len(penalized) > 0:
    print(f"\nPenalized individuals: {len(penalized)} ({len(penalized)/len(ga_history_df)*100:.1f}%)")

    # Reason breakdown
    error_reasons = penalized["reason"].value_counts().head(5)
    print("\nTop failure reasons:")
    for reason, count in error_reasons.items():
        print(f"  - {str(reason)[:80]:80s}: {count} times")

    # Which generations had the most failures?
    failures_by_gen = penalized["generation"].value_counts().sort_index()
    print("\nFailures by generation:")
    for gen, count in failures_by_gen.items():
        pct = (count / len(ga_history_df[ga_history_df["generation"] == gen])) * 100
        print(f"  Gen {gen:2d}: {count:2d} failures ({pct:5.1f}%)")
else:
    error_reasons = pd.Series(dtype="int64")
    failures_by_gen = pd.Series(dtype="int64")
    print("No penalized individuals. All solutions converged to feasible MILP states.")

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

total_evals = len(ga_history_df)
total_feasible = len(ga_history_df[ga_history_df["status"] == "OK"])
success_rate = (total_feasible / total_evals) * 100
best_fitness = ga_ranked_df.iloc[0]["fitness"]
best_cost = ga_ranked_df.iloc[0]["total_cost"]
best_reuse = ga_ranked_df.iloc[0]["reuse_rate"]

print(f"\nGA Execution Summary:")
print(f"  Generations run:       {ga_history_df['generation'].max() + 1}")
print(f"  Population size:       {GA_CONFIG['population_size']}")
print(f"  Total individuals:     {total_evals}")
print(f"  Feasible solutions:    {total_feasible}/{total_evals} ({success_rate:.1f}%)")
print(f"\nBest Design Found:")
print(f"  Fitness score:         {best_fitness:.6f}")
print(f"  Carbon cost:           {best_cost:.2f} kg CO2e")
print(f"  Reclaimed timber:      {best_reuse:.1f}%")
print(f"  Found in generation:   {ga_ranked_df.iloc[0]['generation']}")
print("\nOptimizer Status: COMPLETE AND READY FOR EXPORT")

# -----------------------------------------------------------------------------
# Save extended diagnostics artifacts
# -----------------------------------------------------------------------------
diagnostic_df.to_csv(GA_ANALYSIS_TABLE_DIR / "diagnostic_df_top30.csv", index=False)
diagnostic_source.to_csv(GA_ANALYSIS_TABLE_DIR / "multi_criteria_source_all_feasible.csv", index=False)

# Explicit multi-criteria exports
multi_criteria_cols = [
    "generation",
    "individual",
    "fitness",
    "total_cost",
    "reuse_rate",
    "waste_total",
    "cost_rank",
    "reuse_rank",
    "waste_rank",
    "fitness_rank",
    "overall_score",
]
diagnostic_df[multi_criteria_cols].to_csv(
    GA_ANALYSIS_TABLE_DIR / "multi_criteria_tradeoff_top30.csv", index=False
)

diagnostic_df.nsmallest(len(diagnostic_df), "overall_score")[multi_criteria_cols].to_csv(
    GA_ANALYSIS_TABLE_DIR / "multi_criteria_tradeoff_sorted.csv", index=False
)

cost_winners.to_csv(GA_ANALYSIS_TABLE_DIR / "winners_lowest_carbon.csv", index=False)
reuse_winners.to_csv(GA_ANALYSIS_TABLE_DIR / "winners_highest_reuse.csv", index=False)
waste_winners.to_csv(GA_ANALYSIS_TABLE_DIR / "winners_lowest_waste.csv", index=False)
overall_winners.to_csv(GA_ANALYSIS_TABLE_DIR / "winners_overall_balanced.csv", index=False)

gen_detail.to_csv(GA_ANALYSIS_TABLE_DIR / "generation_detail.csv")
if len(error_reasons) > 0:
    error_reasons.rename("count").to_csv(GA_ANALYSIS_TABLE_DIR / "error_reasons_top5.csv")
if len(failures_by_gen) > 0:
    failures_by_gen.rename("count").to_csv(GA_ANALYSIS_TABLE_DIR / "failures_by_generation.csv")

final_summary = {
    "run_id": GA_ANALYSIS_RUN_ID,
    "analysis_export_dir": str(GA_ANALYSIS_DIR),
    "generations_run": int(ga_history_df["generation"].max() + 1),
    "population_size": int(GA_CONFIG["population_size"]),
    "total_individuals": int(total_evals),
    "feasible_solutions": int(total_feasible),
    "success_rate_percent": float(success_rate),
    "best_fitness": float(best_fitness),
    "best_cost": float(best_cost),
    "best_reuse": float(best_reuse),
    "best_found_generation": int(ga_ranked_df.iloc[0]["generation"]),
}
with open(GA_ANALYSIS_META_DIR / "final_summary.json", "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=2)

print(f"\nExtended diagnostics exported to: {GA_ANALYSIS_DIR}")


# EXPORT

In [ ]:
# Export GA history and best candidate payload
import shutil

ga_history_path = config.GA_DATA_PATH / "c23_ga_history.csv"
ga_ranked_path = config.GA_DATA_PATH / "c23_ga_ranked.csv"
ga_best_design_path = config.GA_DATA_PATH / "c23_ga_best_design.json"

ga_history_df.to_csv(ga_history_path, index=False)
ga_ranked_df.to_csv(ga_ranked_path, index=False)

analysis_export_dir = str(globals().get("GA_ANALYSIS_DIR", ""))
analysis_run_id = str(globals().get("GA_ANALYSIS_RUN_ID", ""))

# Fallback if analysis cell was not run in this kernel session
if not analysis_export_dir:
    fallback_id = f"ID{__import__('datetime').datetime.now():%Y%m%d_%H%M%S}_EXPORT_ONLY"
    GA_ANALYSIS_RUN_ID = fallback_id
    GA_ANALYSIS_DIR = config.GA_DATA_PATH / fallback_id
    GA_ANALYSIS_FIG_DIR = GA_ANALYSIS_DIR / "figures"
    GA_ANALYSIS_TABLE_DIR = GA_ANALYSIS_DIR / "tables"
    GA_ANALYSIS_META_DIR = GA_ANALYSIS_DIR / "meta"
    GA_ANALYSIS_FIG_DIR.mkdir(parents=True, exist_ok=True)
    GA_ANALYSIS_TABLE_DIR.mkdir(parents=True, exist_ok=True)
    GA_ANALYSIS_META_DIR.mkdir(parents=True, exist_ok=True)
    analysis_export_dir = str(GA_ANALYSIS_DIR)
    analysis_run_id = GA_ANALYSIS_RUN_ID

best_payload = {
    "fitness": float(best_overall["fitness"]),
    "status": best_overall["status"],
    "milp_status": best_overall.get("milp_status"),
    "total_cost": float(best_overall.get("total_cost", float("inf"))),
    "reuse_rate": float(best_overall.get("reuse_rate", 0.0)),
    "waste_total": float(best_overall.get("waste_total", float("inf"))),
    "design_params": best_overall["design_params"],
    "normalization_constants": FIXED_NORMALIZATION_CONSTANTS,
    "ga_config": GA_CONFIG,
    "analysis_run_id": analysis_run_id,
    "analysis_export_dir": analysis_export_dir,
}

with open(ga_best_design_path, "w", encoding="utf-8") as f:
    json.dump(best_payload, f, indent=2)

# Also copy core c23 exports into the run-specific analysis folder
analysis_dir_path = Path(analysis_export_dir)
analysis_dir_path.mkdir(parents=True, exist_ok=True)

c23_history_in_run = analysis_dir_path / "c23_ga_history.csv"
c23_ranked_in_run = analysis_dir_path / "c23_ga_ranked.csv"
c23_best_in_run = analysis_dir_path / "c23_ga_best_design.json"

shutil.copy2(ga_history_path, c23_history_in_run)
shutil.copy2(ga_ranked_path, c23_ranked_in_run)
shutil.copy2(ga_best_design_path, c23_best_in_run)

print(f"GA history exported: {ga_history_path}")
print(f"GA ranked table exported: {ga_ranked_path}")
print(f"Best design exported: {ga_best_design_path}")
print(f"Analysis folder exported: {analysis_export_dir}")
print(f"Copied to run folder: {c23_history_in_run}")
print(f"Copied to run folder: {c23_ranked_in_run}")
print(f"Copied to run folder: {c23_best_in_run}")

display(pd.DataFrame([best_payload]).drop(columns=["design_params", "ga_config"]))
